# Query Strings in Express.js

Express parses query strings automatically and exposes them on `req.query`. A query string appears at the end of a URL after a question mark (`?`) and consists of key–value pairs separated by ampersands (`&`).

```
https://api.example.com/search?term=javascript&page=2
                               └──────┬───────┘ └──┬─┘
                                    pair 1      pair 2
```

---

## Basic Implementation

```js
const express = require('express');
const app = express();

// Example URL: http://localhost:3000/search?term=javascript&page=2
app.get('/search', (req, res) => {
  const searchTerm = req.query.term;   // 'javascript'
  const pageNumber = req.query.page;   // '2'  ← still a STRING

  res.json({
    message: 'Query parameters received',
    term: searchTerm,
    page: pageNumber,
  });
});

app.listen(3000, () => console.log('Server running on port 3000'));
```

> **Note:** `req.query` is always an object. If the URL has no query string, it is `{}` — never `undefined`. Individual missing keys are `undefined`.

---

## Advanced Query String Handling

Express uses the [`qs`](https://www.npmjs.com/package/qs) library when the `extended` parser is active, which enables rich structures directly from the URL.

### 1. Arrays

Repeating a key groups the values into an array:

| URL | `req.query` |
|---|---|
| `/products?color=blue&color=red` | `{ color: ['blue', 'red'] }` |
| `/products?color[]=blue&color[]=red` | `{ color: ['blue', 'red'] }` |
| `/products?color[0]=blue&color[1]=red` | `{ color: ['blue', 'red'] }` |
| `/products?color=blue` | `{ color: 'blue' }` ← **string, not array** |

Because a single value does not produce an array, normalize before iterating:

```js
const colors = [].concat(req.query.color ?? []);
// 'blue'            → ['blue']
// ['blue','red']    → ['blue','red']
// undefined         → []
```

### 2. Nested Objects

Bracket notation builds structured data:

```
/users?filter[status]=active&filter[role]=admin
```

```js
req.query.filter // { status: 'active', role: 'admin' }
```

Nesting can go deeper (`?a[b][c]=1`), but `qs` caps depth at **5** by default; anything beyond that is kept as a literal string key.

### 3. Empty and Flag-Style Values

| URL | Result |
|---|---|
| `?debug=` | `{ debug: '' }` |
| `?debug` | `{ debug: '' }` |
| `?debug=false` | `{ debug: 'false' }` ← truthy string! |

There is no such thing as a boolean in a query string. Parse explicitly:

```js
const toBool = (v, fallback = false) =>
  v === undefined ? fallback : ['1', 'true', 'yes', 'on'].includes(String(v).toLowerCase());

const debug = toBool(req.query.debug);
```

### 4. Comma-Separated Values Are Not Split

`?tags=a,b,c` yields the single string `'a,b,c'`. Split it yourself if that's your API contract:

```js
const tags = (req.query.tags ?? '').split(',').filter(Boolean);
```

---

## Configuring the Parser

```js
app.set('query parser', 'extended'); // qs — arrays + nested objects
app.set('query parser', 'simple');   // Node's built-in querystring — flat only
app.set('query parser', false);      // disable parsing entirely; req.query is {}
```

You can also supply a custom function:

```js
const qs = require('qs');

app.set('query parser', (str) =>
  qs.parse(str, {
    depth: 3,           // limit nesting
    arrayLimit: 50,     // indices above this become object keys
    parameterLimit: 100,
    allowPrototypes: false,
  })
);
```

### Express 4 vs Express 5

| | Express 4 | Express 5 |
|---|---|---|
| Default `query parser` | `'extended'` (qs) | `'simple'` (Node `querystring`) |
| `req.query` mutability | Writable / cached | Getter — assigning to it fails |

If you upgrade to Express 5 and nested/bracket parsing suddenly stops working, set `app.set('query parser', 'extended')` explicitly. And instead of mutating `req.query`, attach a cleaned copy:

```js
req.validatedQuery = schema.parse(req.query);
// or res.locals.query = ...
```

---

## Best Practices

### Always sanitize and validate
Values arrive as strings, arrays of strings, or nested objects — never numbers, booleans, or dates. Coerce explicitly before math or database use.

```js
const page = Number.parseInt(req.query.page, 10);
if (!Number.isInteger(page) || page < 1) {
  return res.status(400).json({ error: 'page must be a positive integer' });
}
```

### Provide fallbacks
```js
const limit = Number(req.query.limit) || 10;
```
Careful: `||` also swallows a legitimate `0`. Use `??` when zero is valid:
```js
const offset = req.query.offset ?? 0;
```

### Clamp anything that touches the database
An unbounded `?limit=100000` is a denial-of-service vector.

```js
const limit = Math.min(Math.max(Number(req.query.limit) || 20, 1), 100);
```

### Never interpolate query values into SQL or shell commands
`req.query` is fully attacker-controlled. Use parameterized queries.

```js
// BAD
db.query(`SELECT * FROM users WHERE role = '${req.query.role}'`);
// GOOD
db.query('SELECT * FROM users WHERE role = $1', [req.query.role]);
```

### Guard against type-confusion attacks
An attacker can turn a value you expected to be a string into an object or array: `?email[$ne]=x` becomes `{ email: { $ne: 'x' } }` — a classic NoSQL injection. Assert the shape, don't assume it:

```js
if (typeof req.query.email !== 'string') {
  return res.status(400).json({ error: 'invalid email' });
}
```

Also set `allowPrototypes: false` (the default in `qs`) so `?__proto__[x]=y` can't pollute prototypes.

---

## Validation with a Schema

Hand-rolled parsing gets unwieldy fast. A schema library handles coercion, defaults, and errors in one pass.

```js
const { z } = require('zod');

const searchSchema = z.object({
  term:  z.string().trim().min(1),
  page:  z.coerce.number().int().min(1).default(1),
  limit: z.coerce.number().int().min(1).max(100).default(20),
  sort:  z.enum(['asc', 'desc']).default('asc'),
  tags:  z.union([z.string(), z.array(z.string())])
          .optional()
          .transform((v) => (v == null ? [] : [].concat(v))),
});

app.get('/search', (req, res) => {
  const parsed = searchSchema.safeParse(req.query);
  if (!parsed.success) {
    return res.status(400).json({ errors: parsed.error.flatten().fieldErrors });
  }
  const { term, page, limit, sort, tags } = parsed.data; // properly typed
  res.json({ term, page, limit, sort, tags });
});
```

Reusable as middleware:

```js
const validateQuery = (schema) => (req, res, next) => {
  const parsed = schema.safeParse(req.query);
  if (!parsed.success) return res.status(400).json(parsed.error.flatten());
  req.validatedQuery = parsed.data;
  next();
};

app.get('/search', validateQuery(searchSchema), (req, res) => {
  res.json(req.validatedQuery);
});
```

---

## Encoding Rules

Reserved characters must be percent-encoded, or they will break parsing.

| Character | Encoded |
|---|---|
| space | `%20` (or `+`) |
| `&` | `%26` |
| `=` | `%3D` |
| `?` | `%3F` |
| `#` | `%23` |
| `+` | `%2B` |

Express decodes these for you. `+` is decoded as a space, so a literal plus sign (e.g. in `a+b@mail.com`) **must** be sent as `%2B`.

Build query strings on the client with `URLSearchParams` rather than string concatenation:

```js
const params = new URLSearchParams({ term: 'c++ & rust', page: '2' });
fetch(`/search?${params}`); // term=c%2B%2B+%26+rust&page=2
```

For nested structures, use `qs.stringify` on the client so it matches the server parser:

```js
qs.stringify({ filter: { status: 'active' } }); // filter%5Bstatus%5D=active
```

---

## Query Strings vs. Route Parameters vs. Body

| Feature | Query Strings (`req.query`) | Route Parameters (`req.params`) | Body (`req.body`) |
|---|---|---|---|
| URL syntax | `/search?id=42` | `/user/:id` | not in URL |
| Purpose | Optional filtering, sorting, pagination | Identifying a specific resource | Submitting/changing data |
| Necessity | Optional; handle missing values | Required; route won't match if absent | Depends on endpoint |
| Typical methods | `GET`, `DELETE` | any | `POST`, `PUT`, `PATCH` |
| Visible in logs/history | Yes | Yes | No |
| Cacheable by proxies | Yes (part of the cache key) | Yes | No |
| Size limit | ~2 KB practical URL limit | small | large (configurable) |
| Needs middleware | No | No | Yes (`express.json()`) |

**Rule of thumb:** if removing the value still leaves a meaningful request, it belongs in the query string. Never put secrets, tokens, or passwords there — URLs land in server logs, browser history, and `Referer` headers.

---

## Common Gotchas

- **`req.query` values are read-only in Express 5.** Write derived values to `req.validatedQuery` or `res.locals`.
- **Key order is not guaranteed** and shouldn't be relied on.
- **Keys are case-sensitive.** `?Page=2` will not populate `req.query.page`.
- **Duplicate keys silently become arrays**, which breaks `.trim()` and friends at runtime. Always type-check.
- **`req.query` is not affected by `express.json()`** — that middleware only handles `req.body`.
- **Trailing `?` with nothing after it** produces `{}`, not an error.
- **Route matching ignores the query string.** `app.get('/search?x=1', ...)` is not a valid route definition.

---

## Quick Testing

```js
const request = require('supertest');

it('parses arrays and nested filters', async () => {
  const res = await request(app)
    .get('/search')
    .query({ term: 'js', page: 2 });

  expect(res.body.page).toBe(2);
});
```

Or from the terminal — quote the URL so the shell doesn't eat the `&`:

```bash
curl "http://localhost:3000/search?term=javascript&page=2"
curl -G http://localhost:3000/search --data-urlencode "term=c++ & rust"
```

---

## Cheat Sheet

```js
req.query                          // {} when no query string
req.query.foo                      // undefined when absent
[].concat(req.query.foo ?? [])     // always an array
Number.parseInt(req.query.n, 10)   // always validate the result
req.query.flag === 'true'          // booleans are strings
app.set('query parser', 'extended')// enable nested/bracket parsing
```